# LoRA Fine-Tuning of FLAN-T5 with Hugging Face PEFT

## CNN/DailyMail summarization demo

This modernized notebook replaces the older custom TensorFlow `LoRALayer` implementation with Hugging Face **PEFT**. PEFT inserts and trains LoRA adapters while keeping the pretrained FLAN-T5 parameters frozen.

## 1. Install dependencies

Run this cell before importing Transformers. A runtime restart is only needed when the current session has already imported an incompatible `torchvision`.

In [ ]:
# Remove optional packages that commonly conflict with current
# PyTorch, Transformers, and PEFT installations in Colab.
!pip uninstall -y torchvision torchao
# Removes torchvision and torchao without requesting confirmation.

# torchvision is mainly used for image models and is unnecessary here.
# torchao provides PyTorch model-optimization utilities but can sometimes cause compatibility issues.
# -y automatically confirms removal.

# This line may help resolve environment conflicts, but removing torchvision means later image-based code may require reinstalling it.

# Install the text-model and LoRA dependencies.
# Do not reinstall torch because Colab already provides it.
!pip install -q -U \
    transformers \
    datasets \
    peft \
    accelerate \
    evaluate \
    sacrebleu \
    sentencepiece

# transformers: Provides the model, tokenizer, trainer and training settings.
# datasets: Downloads and processes CNN/DailyMail.
# peft: Provides LoRA and other parameter-efficient fine-tuning methods.
# accelerate: Helps training run efficiently on CPU, GPU or multiple devices.
# evaluate: Provides standard evaluation metrics.
# sacrebleu: Calculates BLEU scores, although this code does not actually use it.
# sentencepiece: Supports tokenization for models such as T5.
# -q: Reduces installation messages.
# -U: Upgrades existing installations.

Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 160.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 7.5 MB/s eta 0:00:00


In [ ]:
import importlib.util
import torch
import transformers
import peft
import datasets
import accelerate

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Datasets:", datasets.__version__)
print("Accelerate:", accelerate.__version__)
print("CUDA available:", torch.cuda.is_available())

print(
    "torchvision installed:",
    importlib.util.find_spec("torchvision") is not None
)

print(
    "torchao installed:",
    importlib.util.find_spec("torchao") is not None
)

PyTorch: 2.11.0+cu128
Transformers: 5.14.1
PEFT: 0.20.0
Datasets: 5.0.1
Accelerate: 1.14.0
CUDA available: True
torchvision installed: False
torchao installed: False


## 2. Check the environment

In [ ]:
import os
import random
import numpy as np
import torch
import transformers
import datasets
import peft

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch: 2.11.0+cu128
Transformers: 5.14.1
Datasets: 5.0.1
PEFT: 0.20.0
CUDA available: True


## 3. Load a manageable CNN/DailyMail subset

In [ ]:
from datasets import load_dataset

raw_dataset = load_dataset("abisee/cnn_dailymail", "3.0.0")

# Each example contains fields such as:

# article: The full news article.
# highlights: The reference summary.
# id: The example identifier.

# The dataset already provides separate train, validation and test splits.

TRAIN_SAMPLES = min(5000, len(raw_dataset["train"])) #Selects up to 5,000 training examples.If the training split contains fewer than 5,000 examples, it uses all available examples.
VALIDATION_SAMPLES = min(500, len(raw_dataset["validation"])) #Selects up to 500 validation examples.
TEST_SAMPLES = min(500, len(raw_dataset["test"])) #Selects up to 500 test examples.

# The three subsets have different purposes:

# Training set: Updates the LoRA parameters.
# Validation set: Selects the best checkpoint.
# Test set: Demonstrates performance on unseen examples.

train_raw = raw_dataset["train"].shuffle(seed=SEED).select(range(TRAIN_SAMPLES))
validation_raw = raw_dataset["validation"].shuffle(seed=SEED).select(range(VALIDATION_SAMPLES))
test_raw = raw_dataset["test"].shuffle(seed=SEED).select(range(TEST_SAMPLES))
#shuffle() prevents the code from always using the original first examples.
# seed=SEED makes the selection reproducible.
# select() extracts the requested rows
print(train_raw)
print("Article preview:", train_raw[0]["article"][:300], "...")
print("Reference summary:", train_raw[0]["highlights"])

README.md:   0%|          | 0.00/15.6k [00:00<?, ?B/s]

3.0.0/train-00000-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00000-of-00003.parquet: downloading bytes:           |  0.00B            

3.0.0/train-00001-of-00003.parquet: reconstructing file:   0%|          |  0.00B /  257MB            

3.0.0/train-00001-of-00003.parquet: downloading bytes:           |  0.00B            

...

3.0.0/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/287113 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/13368 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/11490 [00:00<?, ? examples/s]

Dataset({
    features: ['article', 'highlights', 'id'],
    num_rows: 5000
})
Article preview: By . Anthony Bond . PUBLISHED: . 07:03 EST, 2 March 2013 . | . UPDATED: . 08:07 EST, 2 March 2013 . Three members of the same family who died in a static caravan from carbon monoxide poisoning would have been unconscious 'within minutes', investigators said today. The bodies of married couple John a ...
Reference summary: John and .
Audrey Cook were discovered alongside their daughter, Maureen .
They were found at Tremarle Home Park in Cornwall .
Investigators say the three died of carbon monoxide .
poisoning .


## 4. Load FLAN-T5 and tokenize the dataset

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_NAME = "google/flan-t5-base"
MAX_INPUT_LENGTH = 512
MAX_TARGET_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

print("Base-model parameters:", f"{base_model.num_parameters():,}")

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model: reconstructing file:   0%|          |  0.00B /  792kB            

spiece.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

...

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Base-model parameters: 247,577,856


In [ ]:
def preprocess_batch(examples): #Defines a function that preprocesses a batch of CNN/DailyMail examples.
    inputs = [f"summarize: {article}" for article in examples["article"]] #Adds the instruction summarize: before every article.
    model_inputs = tokenizer(
        inputs,
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    )
    labels = tokenizer(
        text_target=examples["highlights"],
        max_length=MAX_TARGET_LENGTH,
        truncation=True,
    )
    # Converts the input articles into token IDs and attention masks.
    # max_length=MAX_INPUT_LENGTH: Limits the size of each input article.
    # truncation=True: Removes tokens beyond this limit.
    # MAX_INPUT_LENGTH must have been defined earlier.
    #No padding is applied here because the data collator will dynamically pad each batch later.
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_dataset = train_raw.map( #Applies preprocess_batch() to the training subset.
    preprocess_batch,
    batched=True,
    remove_columns=train_raw.column_names,
    desc="Tokenizing training data",
)
validation_dataset = validation_raw.map( #Tokenizes the validation examples.
    preprocess_batch,
    batched=True,
    remove_columns=validation_raw.column_names,
    desc="Tokenizing validation data",
)
test_dataset = test_raw.map( #Tokenizes the test examples.
    preprocess_batch,
    batched=True,
    remove_columns=test_raw.column_names,
    desc="Tokenizing test data",
)

print(train_dataset.column_names)
print(train_dataset[0].keys())

Tokenizing training data:   0%|          | 0/5000 [00:00<?, ? examples/s]

Tokenizing validation data:   0%|          | 0/500 [00:00<?, ? examples/s]

Tokenizing test data:   0%|          | 0/500 [00:00<?, ? examples/s]

['input_ids', 'attention_mask', 'labels']
dict_keys(['input_ids', 'attention_mask', 'labels'])


## 5. Add LoRA adapters with PEFT

In [ ]:
from peft import LoraConfig, TaskType, get_peft_model
# Imports the PEFT components required for LoRA.
# LoraConfig: Defines the LoRA configuration.
# TaskType: Specifies the type of model task.
# get_peft_model: Inserts LoRA layers into the base model.
lora_config = LoraConfig( #Begins creating the LoRA configuration.
    task_type=TaskType.SEQ_2_SEQ_LM,
    inference_mode=False, #Places LoRA in training mode. If it were True, the adapter would be prepared only for inference rather than fine-tuning.
    r=8, #Sets the LoRA rank to 8.
    lora_alpha=16, #Sets the scaling factor applied to LoRA updates. The effective scaling is commonly related to: lora_alpha/r = 16/8 = 2
    lora_dropout=0.05, #Applies a 5% dropout rate to the LoRA path during training. This can help reduce overfitting.
    target_modules=["q", "v"], #Adds LoRA adapters to the query (q) and value (v) projection layers in the T5 attention modules. These projections help the attention mechanism decide:
                                # - What information to search for.
                                # - What information to pass forward.
                                # Targeting q and v is a common LoRA configuration for T5 models.
    bias="none", #Leaves all original bias parameters unchanged. #Only the LoRA parameters are trained.
)
# States that the model performs sequence-to-sequence language modelling.
# This is appropriate for T5 and tasks such as:
# Summarization
# Translation
# Text rewriting

model = get_peft_model(base_model, lora_config) #Wraps the previously loaded base_model with LoRA adapters. The resulting model contains:
                                                # 1. The frozen original model parameters.
                                                # 2. Small trainable LoRA parameters.
                                                # base_model must have been created earlier
model.print_trainable_parameters()

trainable params: 884,736 || all params: 248,462,592 || trainable%: 0.3561


## 6. Configure and run training

In [ ]:
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

# Enables BF16 calculations when:

# A CUDA GPU is available. The GPU supports the BF16 format.
# BF16 reduces memory usage while retaining a wide numerical range.

USE_FP16 = torch.cuda.is_available() and not USE_BF16

# Uses FP16 when a CUDA GPU exists but BF16 is unavailable.
# This makes the choice mutually exclusive:
# Supported GPU: BF16
# Older compatible GPU: FP16
# CPU: Neither

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True,
    label_pad_token_id=-100,
    return_tensors="pt",
)
# Creates an object that constructs padded PyTorch batches.

# tokenizer=tokenizer: Supplies the appropriate padding token.
# model=model: Helps prepare decoder inputs.
# padding=True: Pads each batch to its longest sequence.
# label_pad_token_id=-100: Marks target padding so it is ignored by the loss.
# return_tensors="pt": Returns PyTorch tensors.

training_args = Seq2SeqTrainingArguments( #Begins defining the sequence-to-sequence training settings.
    output_dir="./chapter4_demo1_lora_summarization", #Stores checkpoints and training outputs in this directory.
    num_train_epochs=2, #Trains the LoRA parameters for two complete passes through the training subset.
    learning_rate=5e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4, #Accumulates gradients over four batches before updating the LoRA parameters. With one device, the effective training batch size is: 4×4=16
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    predict_with_generate=True, #Configures the trainer to generate summaries during prediction-based evaluation.
    generation_max_length=MAX_TARGET_LENGTH,
    generation_num_beams=4,
    fp16=USE_FP16, #Activates the mixed-precision format selected earlier.
    bf16=USE_BF16,
    report_to="none", #Disables automatic external experiment logging.
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss", #Uses validation loss to select the best checkpoint.
    greater_is_better=False,
    save_total_limit=2,
)

trainer = Seq2SeqTrainer( #Creates the object responsible for training and evaluation.
    model=model, #Provides the LoRA-wrapped sequence-to-sequence model.
    args=training_args, #Provides the training configuration.
    train_dataset=train_dataset,
    eval_dataset=validation_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
)

trainer.train() #Starts LoRA fine-tuning.

Epoch,Training Loss,Validation Loss
1,7.930004,1.596807
2,7.851630,1.597391


TrainOutput(global_step=626, training_loss=7.800881139005716, metrics={'train_runtime': 467.1967, 'train_samples_per_second': 21.404, 'train_steps_per_second': 1.34, 'total_flos': 6874145702645760.0, 'train_loss': 7.800881139005716, 'epoch': 2.0})

## 7. Evaluate and inspect generated summaries

In [ ]:
metrics = trainer.evaluate()

# Evaluates the trained model on the validation dataset.
# Despite the name metrics, the output will mainly contain values such as:
# - eval_loss
# - eval_runtime
# - eval_samples_per_second
# - eval_steps_per_second
# It will not automatically calculate ROUGE, BLEU or other text-quality metrics because no compute_metrics function was supplied.

print(metrics)

Training Loss,Validation Loss,Epoch
7.851630,1.596807,2


{'eval_loss': 1.5968068838119507}


In [ ]:
model.eval() #Places the trained model in evaluation mode. This disables training-specific behaviour such as dropout.
device = next(model.parameters()).device #Finds the device on which the model is currently stored.

for i in range(min(5, len(test_raw))):
    article = test_raw[i]["article"]
    reference = test_raw[i]["highlights"]
    # The original test article.
    # Its human-written reference summary.
    inputs = tokenizer(
        f"summarize: {article}",
        return_tensors="pt",
        max_length=MAX_INPUT_LENGTH,
        truncation=True,
    ).to(device)
    #Adds the summarization instruction, tokenizes the article and moves the tensors to the model’s device.
    with torch.inference_mode():
        generated = model.generate(
            **inputs,
            max_new_tokens=MAX_TARGET_LENGTH,
            num_beams=4,
            early_stopping=True,
        )
    # Generates a summary.
    # **inputs: Passes the input IDs and attention mask.
    # max_new_tokens: Limits the number of generated tokens.
    # num_beams=4: Uses four-path beam search.
    # early_stopping=True: Stops beam search when completed candidates are found.
    #Disables gradient calculation because the model is only generating summaries.
    prediction = tokenizer.decode(generated[0], skip_special_tokens=True)
    # Converts the generated token IDs into a readable summary.
    print(f"\nExample {i + 1}")
    print("Reference:", reference)
    print("Prediction:", prediction)
    print("-" * 100)


Example 1
Reference: CNN's Dr. Sanjay Gupta says we should legalize medical marijuana now .
He says he knows how easy it is do nothing "because I did nothing for too long"
Prediction: I see signs of a revolution everywhere. I see it in the op-ed pages of the newspapers, and on the state ballots in nearly half the country . I see it in the eyes of sterling scientists, previously reluctant to dip a toe into this heavily stigmatized world . I see it in the faces of good parents, uprooting their lives to get medicine for their children .
----------------------------------------------------------------------------------------------------

Example 2
Reference: Child has amassed thousands of Twitter followers with 'gang life' photos .
In one video he points gun at camera as adults look on unfazed .
His tweets have prompted backlash with calls for intervention .
Prediction: Baby-faced boy from Memphis, Tennessee, poses with guns, cash, and bags of marijuana . He has amassed more than 3,000 fo

## 8. Save the LoRA adapter

In [ ]:
ADAPTER_DIR = "./chapter4_demo1_lora_adapter"

#Saves the LoRA adapter configuration and weights. Because model is a PEFT model, this primarily saves the small adapter rather than another complete copy of the base model.

model.save_pretrained(ADAPTER_DIR) #Saves the tokenizer files in the same directory.
tokenizer.save_pretrained(ADAPTER_DIR)
print("Adapter saved to", ADAPTER_DIR)

Adapter saved to ./chapter4_demo1_lora_adapter


Overall workflow

Code 6 performs parameter-efficient summarization fine-tuning:

* Install the required libraries.
* Load CNN/DailyMail.
* Select 5,000 training, 500 validation and 500 test examples.
* Tokenize articles and reference summaries.
* Add LoRA adapters to the T5 attention layers.
* Train only the small LoRA parameter set.
* Select the best checkpoint using validation loss.
* Generate summaries for five unseen test articles.
* Save the adapter and tokenizer.